In [1]:
from platform import python_version
print(python_version())

3.11.14


### Open Access

- disease 
- drug_mechanism_of_action 
- evidence_cancer_gene_census 
- evidence_reactome 
- interactions 
- pharmacogenomics 
- study

https://www.opentargets.org/

find a gene:
https://platform.opentargets.org/target/ENSG00000153395/associations

download:
https://platform.opentargets.org/downloads

In [ ]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"


if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)


from libs.Basic import pdwritecsv, pdreadcsv, create_dir, write_txt
from libs.enricher_lib import enricheR
from libs.config_lib import Config
from libs.MTD_lib import MTD
from libs.tcga_gdc_lib import GDC

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root_ot { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'

disease = PSI_ID

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
enr = enricheR(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
               root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
               case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
               std_filename=std_filename, std_filename_list=std_filename_list,
               geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
               tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
               LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
               num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
               min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
               saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", enr.root0, enr.root_disease)
case = case_list[0]
print(">>>", case)

enr.cfg.set_default_best_lfc_cutoff(enr.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = enr.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(enr.echo_parameters())


Start opening tables ....
Building synonym dictionary ...

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/TCGA-CESC
>>> Tumor
>>> case Tumor
	DEGs 20006
		Up (#10358)
		Dw (#9648)

Up-regulated per biotype
                               biotype     n
0                            IG_C_gene    12
1                      IG_C_pseudogene     4
2                            IG_D_gene     1
3                            IG_J_gene     9
4                      IG_J_pseudogene     1
5                            IG_V_gene   124
6                      IG_V_pseudogene    50
7                              Mt_tRNA    17
8                                  TEC   112
9                            TR_C_gene     5
10                           TR_D_gene     1
11                           TR_J_gene    10
12                           TR_V_gene    69
13                     TR_V_pseudogene     1
14                              lncRNA  3193
15                               miRNA   

In [5]:
enr.root_kegg, enr.kegg_fname

(PosixPath('/home/flavio/uv/perturb_agent/data/colab/kegg'),
 'kegg_pathways.tsv')

In [6]:
enr.enr_db_list

['Reactome_Pathways_2024']

In [7]:
print(len(enr.gene.df_my_gene))
enr.gene.df_my_gene.head(2)

27223


,entrezid,symbol,geneid,name,synonyms,other_names,ec_enzyme,refseq_gen,refseq_prot,refseq_rna,...,acessions_rna,acessions_trans,map_location,dic_panther,ortholog,dic_uniprot,swissprot,wikipedia,refseq_gen,refseq_rna
0,ENSG00000121410,A1BG,1,alpha-1-B glycoprotein,"['A1B', 'ABG', 'GAB', 'HYST2477']","['HEL-S-163pA', 'alpha-1B-glycoprotein', 'epididymis secretory sperm binding...",NaN,NaN,NP_570602.2,NaN,...,"['AA484435.1', 'AB073611.1', 'AF414429.1', 'AI022193.1', 'AK055885.1', 'AK05...","[{'protein': 'AAH35719.1', 'rna': 'BC035719.1'}, {'protein': 'BAG51591.1', '...",19q13.43,"{'HGNC': '5', '_license': 'http://pantherdb.org/tou.jsp', 'ortholog': [{'MGI...","[{'MGI': '2152878', 'ortholog_type': 'LDO', 'panther_family': 'PTHR11738', '...","{'Swiss-Prot': 'P04217', 'TrEMBL': ['M0R009', 'V9HWD8']}",P04217,{'url_stub': 'A1BG (gene)'},"['NC_000019.10', 'NC_060943.1']",NM_130786.4
1,ENSG00000268895,A1BG-AS1,503538,A1BG antisense RNA 1,"['A1BG-AS', 'A1BGAS', 'NCRNA00181']","['A1BG antisense RNA (non-protein coding)', 'A1BG antisense RNA 1 (non-prote...",NaN,NaN,NaN,NaN,...,"['AK027222.1', 'BC006144.1', 'BC040926.1', 'NR_015380.2']",NaN,19q13.43,NaN,NaN,NaN,NaN,NaN,"['NC_000019.10', 'NC_060943.1']",NR_015380.2


### Keneddy Pathway

https://www.wikipathways.org/pathways/WP3933.html

![Kennedy Pathway](../figures/WP3933_kennety_pathway.svg)

### Open Targets

```Bash
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.03/output/evidence_cancer_gene_census .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.03/output/drug_mechanism_of_action .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.03/output/pharmacogenomics .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.03/output/study .
rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.03/output/evidence_reactome .

rsync -rpltvz --delete rsync.ebi.ac.uk::pub/databases/opentargets/platform/25.03/output/target .

colab/
    open_targets/
    ├── target/
    ├── disease/
    ├── drug_mechanism_of_action/
    ├── evidence_cancer_gene_census/
    ├── evidence_reactome/
    ├── interactions/
    ├── pharmacogenomics/
    └── study/
```

In [21]:
from pathlib import Path
import duckdb
import pandas as pd

root_ot = root_colab / 'open_targets'

### Is target ok?

In [22]:
from pathlib import Path

target_dir =  root_ot / "target"

print("exists:", target_dir.exists())
print("is dir:", target_dir.is_dir())

for p in target_dir.rglob("*"):
    print(p)

exists: True
is dir: True
/home/flavio/uv/perturb_agent/data/colab/open_targets/target/part-00000-4c6c5f36-b7cf-4ea1-804c-33e0c7b43fad-c000.snappy.parquet
/home/flavio/uv/perturb_agent/data/colab/open_targets/target/part-00199-4c6c5f36-b7cf-4ea1-804c-33e0c7b43fad-c000.snappy.parquet
/home/flavio/uv/perturb_agent/data/colab/open_targets/target/part-00095-4c6c5f36-b7cf-4ea1-804c-33e0c7b43fad-c000.snappy.parquet
/home/flavio/uv/perturb_agent/data/colab/open_targets/target/part-00029-4c6c5f36-b7cf-4ea1-804c-33e0c7b43fad-c000.snappy.parquet
/home/flavio/uv/perturb_agent/data/colab/open_targets/target/part-00020-4c6c5f36-b7cf-4ea1-804c-33e0c7b43fad-c000.snappy.parquet
/home/flavio/uv/perturb_agent/data/colab/open_targets/target/part-00136-4c6c5f36-b7cf-4ea1-804c-33e0c7b43fad-c000.snappy.parquet
/home/flavio/uv/perturb_agent/data/colab/open_targets/target/part-00193-4c6c5f36-b7cf-4ea1-804c-33e0c7b43fad-c000.snappy.parquet
/home/flavio/uv/perturb_agent/data/colab/open_targets/target/part-00061

In [23]:
print("root exists:", root_ot.exists())
print("root is dir:", root_ot.is_dir())

print("\nTop-level content:")
for p in sorted(root_ot.iterdir()):
    print(
        "DIR " if p.is_dir() else "FILE",
        p.name
    )

root exists: True
root is dir: True

Top-level content:
DIR  disease
DIR  drug_mechanism_of_action
DIR  evidence_cancer_gene_census
DIR  evidence_reactome
FILE index.html
DIR  interactions
DIR  pharmacogenomics
FILE robots.txt
DIR  study
DIR  target


In [30]:
class OpenTargetsLocal:
    def __init__(self, root_ot: Path | Path):
        self.root_ot = root_ot
        self.con = duckdb.connect()

        self.tables = {
            "target": self.root_ot / "target",
            "disease": self.root_ot / "disease",
            "drug_moa": self.root_ot / "drug_mechanism_of_action",
            "cgc": self.root_ot / "evidence_cancer_gene_census",
            "reactome": self.root_ot / "evidence_reactome",
            "interactions": self.root_ot / "interactions",
            "pharmacogenomics": self.root_ot / "pharmacogenomics",
            "study": self.root_ot / "study",
        }

        self._create_views()

    def _glob(self, path: Path) -> str:
        return str(path / "**" / "*.parquet")

    def _create_views(self):
        for name, path in self.tables.items():
            if path.exists():
                self.con.execute(
                    f"""
                    CREATE OR REPLACE VIEW {name} AS
                    SELECT * FROM read_parquet('{self._glob(path)}')
                    """
                )
            else:
                print(f"Warning: missing table directory: {path}")

    def schema(self, table: str) -> pd.DataFrame:
        return self.con.execute(f"DESCRIBE {table}").df()
    
    # Resolve gene symbol or Ensembl ID
    def resolve_target(self, gene_or_ensembl: str) -> pd.DataFrame:
        q = gene_or_ensembl.strip()

        sql = """
        SELECT
            id AS target_id,
            approvedSymbol AS symbol,
            approvedName AS name,
            biotype
        FROM target
        WHERE
            id = ?
            OR upper(approvedSymbol) = upper(?)
        """

        return self.con.execute(sql, [q, q]).df()
    

    def show_schema_summary(self):
        for table in [
            "target",
            "disease",
            "drug_moa",
            "cgc",
            "reactome",
            "interactions",
            "pharmacogenomics",
            "study",
        ]:
            print("\n###", table)
            display(self.schema(table))

In [31]:
ot = OpenTargetsLocal(root_ot=root_ot)

ot.resolve_target("TP53")

,target_id,symbol,name,biotype
0,ENSG00000141510,TP53,tumor protein p53,protein_coding


### Find a gene

In [32]:
ot = OpenTargetsLocal(root_ot)

print(ot.con.execute("SHOW TABLES").df())

for table in ["target", "disease", "drug_moa", "cgc", "reactome", "interactions", "pharmacogenomics", "study"]:
    print("\n###", table)
    print(ot.con.execute(f"SELECT COUNT(*) AS n FROM {table}").df())

               name
0               cgc
1           disease
2          drug_moa
3      interactions
4  pharmacogenomics
5          reactome
6             study
7            target

### target
       n
0  78766

### disease
       n
0  47030

### drug_moa
      n
0  6505

### cgc
       n
0  91572

### reactome
       n
0  10748

### interactions
          n
0  14618053

### pharmacogenomics
       n
0  33080

### study
         n
0  2001827


In [33]:
ot.show_schema_summary()


### target


,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,approvedSymbol,VARCHAR,YES,None,None,None
2,biotype,VARCHAR,YES,None,None,None
3,transcriptIds,VARCHAR[],YES,None,None,None
4,canonicalTranscript,"STRUCT(id VARCHAR, chromosome VARCHAR, ""start"" BIGINT, ""end"" BIGINT, strand ...",YES,None,None,None
5,canonicalExons,VARCHAR[],YES,None,None,None
6,genomicLocation,"STRUCT(chromosome VARCHAR, ""start"" BIGINT, ""end"" BIGINT, strand INTEGER)",YES,None,None,None
7,alternativeGenes,VARCHAR[],YES,None,None,None
8,approvedName,VARCHAR,YES,None,None,None
9,go,"STRUCT(id VARCHAR, ""source"" VARCHAR, evidence VARCHAR, aspect VARCHAR, geneP...",YES,None,None,None



### disease


,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,code,VARCHAR,YES,None,None,None
2,name,VARCHAR,YES,None,None,None
3,description,VARCHAR,YES,None,None,None
4,dbXRefs,VARCHAR[],YES,None,None,None
5,parents,VARCHAR[],YES,None,None,None
6,exactSynonyms,VARCHAR[],YES,None,None,None
7,relatedSynonyms,VARCHAR[],YES,None,None,None
8,narrowSynonyms,VARCHAR[],YES,None,None,None
9,broadSynonyms,VARCHAR[],YES,None,None,None



### drug_moa


...

,column_name,column_type,null,key,default,extra
0,sourceDatabase,VARCHAR,YES,None,None,None
1,targetA,VARCHAR,YES,None,None,None
2,intA,VARCHAR,YES,None,None,None
3,intABiologicalRole,VARCHAR,YES,None,None,None
4,targetB,VARCHAR,YES,None,None,None
5,intB,VARCHAR,YES,None,None,None
6,intBBiologicalRole,VARCHAR,YES,None,None,None
7,speciesA,"STRUCT(mnemonic VARCHAR, scientificName VARCHAR, taxonId BIGINT)",YES,None,None,None
8,speciesB,"STRUCT(mnemonic VARCHAR, scientificName VARCHAR, taxonId BIGINT)",YES,None,None,None
9,count,BIGINT,YES,None,None,None



### pharmacogenomics


,column_name,column_type,null,key,default,extra
0,datasourceId,VARCHAR,YES,None,None,None
1,datasourceVersion,VARCHAR,YES,None,None,None
2,datatypeId,VARCHAR,YES,None,None,None
3,directionality,VARCHAR,YES,None,None,None
4,evidenceLevel,VARCHAR,YES,None,None,None
5,genotype,VARCHAR,YES,None,None,None
6,genotypeAnnotationText,VARCHAR,YES,None,None,None
7,genotypeId,VARCHAR,YES,None,None,None
8,haplotypeFromSourceId,VARCHAR,YES,None,None,None
9,haplotypeId,VARCHAR,YES,None,None,None



### study


,column_name,column_type,null,key,default,extra
0,studyId,VARCHAR,YES,None,None,None
1,geneId,VARCHAR,YES,None,None,None
2,projectId,VARCHAR,YES,None,None,None
3,studyType,VARCHAR,YES,None,None,None
4,traitFromSource,VARCHAR,YES,None,None,None
5,traitFromSourceMappedIds,VARCHAR[],YES,None,None,None
6,biosampleFromSourceId,VARCHAR,YES,None,None,None
7,pubmedId,VARCHAR,YES,None,None,None
8,publicationTitle,VARCHAR,YES,None,None,None
9,publicationFirstAuthor,VARCHAR,YES,None,None,None


### Then test target-specific retrieval with TP53:

In [ ]:
symbol = 'TP53'
symbol = 'PCYT2'
symbol = 'CCNB2'
symbol = 'BLM'

# Em adenocarcinoma pancreático, o gene KRAS mutado ativa uma proteína chamada KLF5, que coordena simultaneamente a desregulação de quatro enzimas da via de síntese de membranas celulares — 
# criando um perfil metabólico identificável por biópsia com implicações prognósticas e terapêuticas.
symbol = 'KLF5'

target = ot.resolve_target(symbol)
target_id = target.iloc[0]["target_id"]

target_id

'ENSG00000197299'

### Test Cancer Gene Census

In [49]:
df_cgc = ot.con.execute(
    """
    SELECT
        c.*,
        d.name AS disease_name, d.synonyms, d.ontology
    FROM cgc c
    LEFT JOIN disease d
        ON c.diseaseId = d.id
    WHERE c.targetId = ?
    ORDER BY c.score DESC NULLS LAST
    LIMIT 50
    """,
    [target_id]
).df()

print(len(df_cgc), 'diseases found.')
df_cgc.head(2).T

50 diseases found.


,0,1
targetId,ENSG00000197299,ENSG00000197299
id,f172bc2280e27ba2182f135e521143106ba7e2cd,ec84642a46e4d911da0981988e6e024b1dab6c0f
targetFromSourceId,ENSG00000197299,ENSG00000197299
diseaseFromSourceMappedId,EFO_0001663,MONDO_0002529
datasourceId,cancer_gene_census,cancer_gene_census
datatypeId,somatic_mutation,somatic_mutation
diseaseFromSourceId,None,None
literature,"[21307934, 26000489, 31580249, 31176623, 31874108]","[24662767, 25589618, 25303977, 28481359, 21984974]"
mutatedSamples,"[{'functionalConsequenceId': 'SO_0001583', 'numberMutatedSamples': 110, 'num...","[{'functionalConsequenceId': 'SO_0001583', 'numberMutatedSamples': 15, 'numb..."
resourceScore,1.0,1.0


### Test Reactome evidence

In [ ]:
df_reactome = ot.con.execute(
    """
    SELECT
        r.*,
        d.name AS disease_name
    FROM reactome r
    LEFT JOIN disease d
        ON r.diseaseId = d.id
    WHERE r.targetId = ?
    ORDER BY r.score DESC NULLS LAST
    LIMIT 50
    """,
    [target_id]
).df()

print(len(df_reactome))
df_reactome.head(3).T

,0,1,2
id,208e4524a53ea2738caf4af2d17b09314153748b,a3c6f92e245effabbc42949832ad761b23da3080,d06c5636d984df04b16762b4aab758b2746b1b29
targetFromSourceId,P54132,P54132,P54132
diseaseFromSourceMappedId,EFO_0000311,EFO_0000311,EFO_0000311
datasourceId,reactome,reactome,reactome
datatypeId,affected_pathway,affected_pathway,affected_pathway
diseaseFromSource,cancer,cancer,cancer
diseaseFromSourceId,DOID:162,DOID:162,DOID:162
literature,"[19369211, 29165669, 27899578, 32732220]",[16793542],"[19369211, 29165669, 27899578, 32732220]"
pathways,"[{'id': 'R-HSA-9701192', 'name': 'Defective homologous recombination repair ...","[{'id': 'R-HSA-9709603', 'name': 'Impaired BRCA2 binding to PALB2'}]","[{'id': 'R-HSA-9701192', 'name': 'Defective homologous recombination repair ..."
reactionId,R-HSA-9701199,R-HSA-9709601,R-HSA-9701199


### drug mechanism of action

In [52]:
display(ot.schema('drug_moa'))

,column_name,column_type,null,key,default,extra
0,actionType,VARCHAR,YES,None,None,None
1,mechanismOfAction,VARCHAR,YES,None,None,None
2,chemblIds,VARCHAR[],YES,None,None,None
3,targetName,VARCHAR,YES,None,None,None
4,targetType,VARCHAR,YES,None,None,None
5,targets,VARCHAR[],YES,None,None,None
6,references,"STRUCT(""source"" VARCHAR, ids VARCHAR[], urls VARCHAR[])[]",YES,None,None,None


In [54]:
df_drug = ot.con.execute(
    """
    SELECT *
    FROM drug_moa
    WHERE CAST(targets AS VARCHAR) ILIKE ?
    """,
    [f"%{target_id}%"]
).df()

print(len(df_drug))
df_drug.head(3).T

0


""
actionType
mechanismOfAction
chemblIds
targetName
targetType
targets
references


### interactions

In [55]:
ot.schema("interactions")

,column_name,column_type,null,key,default,extra
0,sourceDatabase,VARCHAR,YES,None,None,None
1,targetA,VARCHAR,YES,None,None,None
2,intA,VARCHAR,YES,None,None,None
3,intABiologicalRole,VARCHAR,YES,None,None,None
4,targetB,VARCHAR,YES,None,None,None
5,intB,VARCHAR,YES,None,None,None
6,intBBiologicalRole,VARCHAR,YES,None,None,None
7,speciesA,"STRUCT(mnemonic VARCHAR, scientificName VARCHAR, taxonId BIGINT)",YES,None,None,None
8,speciesB,"STRUCT(mnemonic VARCHAR, scientificName VARCHAR, taxonId BIGINT)",YES,None,None,None
9,count,BIGINT,YES,None,None,None


In [57]:
df_int = ot.con.execute(
    """
    SELECT *
    FROM interactions
    WHERE targetA = ?
       OR targetB = ?
    LIMIT 100
    """,
    [target_id, target_id]
).df()

print(len(df_int))
df_int.head(3).T

100


,0,1,2
sourceDatabase,intact,intact,intact
targetA,ENSG00000197299,ENSG00000197299,ENSG00000141510
intA,P54132,P54132,P04637
intABiologicalRole,unspecified role,unspecified role,unspecified role
targetB,ENSG00000196584,ENSG00000100603,ENSG00000197299
intB,O43543,Q13573,P54132
intBBiologicalRole,unspecified role,unspecified role,unspecified role
speciesA,"{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}","{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}","{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}"
speciesB,"{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}","{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}","{'mnemonic': 'human', 'scientificName': 'Homo sapiens', 'taxonId': 9606}"
count,1,1,1
